# 04 — Classification
Tests whether the consensus labels can be predicted from the text with TF-IDF classifiers.

| Task | Label | Text fields |
|---|---|---|
| 1 | Expected Response Function | previous utterance |
| 2 | Actual Function | previous utterance + response |
| 3 | Fulfillment | context + previous utterance + response |

Each field gets its own word (1–2-gram) and character (3–5-gram) TF-IDF features. Mismatch Type is not classified.

Input: `final_annotated_dataset.csv` (from notebook 03).

In [ ]:
import os, warnings, logging

# keep the notebook output clean (hide download bars / Hugging Face warnings)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
SEED = 42
N_FOLDS = 5

## Data

In [ ]:
df = pd.read_csv("final_annotated_dataset.csv", dtype=str, keep_default_na=False)
for c in ["context", "previous_utterance", "response"]:
    df[c] = df[c].str.strip()

TASKS = {"Expected Response Function": ["previous_utterance"],
         "Actual Function": ["previous_utterance", "response"],
         "Fulfillment": ["context", "previous_utterance", "response"]}

print(f"{len(df):,} labelled pairs from {df['dialogue_id'].nunique():,} dialogues\n")
for t in TASKS:
    vc = df[t].value_counts()
    print(t, "->", ", ".join(f"{k}: {v}" for k, v in vc.items()))

## Evaluation protocol
* Stratified 5-fold cross-validation, grouped by `dialogue_id` (no dialogue appears in both training and test folds); seed 42.
* Main metric: Macro-F1; accuracy and weighted-F1 are also reported as mean ± standard deviation over the folds.
* Baselines: majority class and stratified random guessing.
* Models: Logistic Regression, Linear SVM, Complement Naive Bayes (default hyper-parameters, `class_weight="balanced"` where supported) and, if the encoder can be downloaded, sentence embeddings (all-MiniLM-L6-v2) + Logistic Regression.

In [ ]:
def make_features():
    # Words capture meaning (e.g. "please", "what"); characters capture spelling variants and short forms ("I'll").
    # Both views are standard for short-text classification (Cavnar & Trenkle, 1994; Manning et al., 2008).
    return FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=30000)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True, max_features=30000)),
    ])

def fields_features(cols):
    return ColumnTransformer([(c, make_features(), c) for c in cols])

def build(name, cols):
    if name == "Logistic Regression":
        return Pipeline([("f", fields_features(cols)), ("c", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
    if name == "Linear SVM":
        return Pipeline([("f", fields_features(cols)), ("c", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=SEED))])
    if name == "Complement Naive Bayes":
        return Pipeline([("f", fields_features(cols)), ("c", ComplementNB(alpha=0.5))])
    if name == "Majority baseline":
        return DummyClassifier(strategy="most_frequent")
    if name == "Stratified-random baseline":
        return DummyClassifier(strategy="stratified", random_state=SEED)
    raise ValueError(name)

MODEL_NAMES = ["Majority baseline", "Stratified-random baseline",
               "Logistic Regression", "Linear SVM", "Complement Naive Bayes"]

# Optional 4th model (needs internet to download the encoder; skipped automatically if unavailable)
try:
    from sentence_transformers import SentenceTransformer
    _encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    HAVE_EMBEDDINGS = True
except Exception:
    HAVE_EMBEDDINGS = False
print("Sentence-embedding model available:", HAVE_EMBEDDINGS)

In [ ]:
def run_cv(task, cols, model_names, data=df):
    y = data[task].values
    groups = data["dialogue_id"].values
    X = data[cols]
    joined = data[cols].agg(" || ".join, axis=1).values          # only used by the embedding model
    skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_rows, oof = [], {m: np.empty(len(data), dtype=object) for m in model_names}
    for k, (tr, te) in enumerate(skf.split(X, y, groups)):
        assert set(groups[tr]).isdisjoint(groups[te])         # no dialogue is in both train and test
        for m in model_names:
            if m == "Sentence Embeddings + LR":
                clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)
                clf.fit(_encoder.encode(list(joined[tr]), normalize_embeddings=True), y[tr])
                pred = clf.predict(_encoder.encode(list(joined[te]), normalize_embeddings=True))
            elif m.endswith("baseline"):
                clf = build(m, cols).fit(np.zeros((len(tr), 1)), y[tr])   # baselines ignore the text
                pred = clf.predict(np.zeros((len(te), 1)))
            else:
                clf = build(m, cols).fit(X.iloc[tr], y[tr])
                pred = clf.predict(X.iloc[te])
            oof[m][te] = pred
            fold_rows.append({"Task": task, "Model": m, "Fold": k + 1,
                              "Accuracy": accuracy_score(y[te], pred),
                              "Macro-F1": f1_score(y[te], pred, average="macro", zero_division=0),
                              "Weighted-F1": f1_score(y[te], pred, average="weighted", zero_division=0)})
    return pd.DataFrame(fold_rows), oof

if HAVE_EMBEDDINGS:
    MODEL_NAMES.append("Sentence Embeddings + LR")

all_folds, all_oof = [], {}
for task, col in TASKS.items():
    folds, oof = run_cv(task, col, MODEL_NAMES)
    all_folds.append(folds); all_oof[task] = oof
folds_df = pd.concat(all_folds, ignore_index=True)
print("Finished", N_FOLDS, "folds ×", len(MODEL_NAMES), "models ×", len(TASKS), "tasks")

## Results

In [ ]:
def summarise(folds):
    g = folds.groupby(["Task", "Model"])[["Accuracy", "Macro-F1", "Weighted-F1"]].agg(["mean", "std"])
    out = pd.DataFrame(index=g.index)
    for metric in ["Accuracy", "Macro-F1", "Weighted-F1"]:
        out[metric] = g[(metric, "mean")].round(3).astype(str) + " ± " + g[(metric, "std")].round(3).astype(str)
    out["_sort"] = g[("Macro-F1", "mean")]
    return out.reset_index()

summary = summarise(folds_df)
for task in TASKS:
    print(f"\n=== {task} ===")
    display(summary[summary.Task == task].sort_values("_sort", ascending=False).drop(columns=["Task", "_sort"]).reset_index(drop=True))

## Per-class results and confusion matrices
Pooled out-of-fold predictions of the best model per task; confusion-matrix rows are percentages of each consensus class.

In [ ]:
real_models = [m for m in MODEL_NAMES if not m.endswith("baseline")]
best = {}
for task in TASKS:
    means = folds_df[(folds_df.Task == task) & folds_df.Model.isin(real_models)].groupby("Model")["Macro-F1"].mean()
    best[task] = means.idxmax()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))
per_class_tables = []
for ax, task in zip(axes, TASKS):
    y = df[task].values; pred = all_oof[task][best[task]]
    labels = sorted(set(y))
    print(f"\n=== {task} — best model: {best[task]} ===")
    print(classification_report(y, pred, labels=labels, zero_division=0, digits=3))
    rep = pd.DataFrame(classification_report(y, pred, labels=labels, zero_division=0, output_dict=True)).T.loc[labels + ["macro avg"]]
    rep.insert(0, "Task", task); rep.insert(1, "Best model", best[task])
    per_class_tables.append(rep.reset_index().rename(columns={"index": "Class"}))
    cm = confusion_matrix(y, pred, labels=labels); cmn = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=40, ha="right"); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cmn[i, j]*100:.0f}%\n(n={cm[i, j]})", ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.55 else "black")
    ax.set_title(f"{task}\n{best[task]}"); ax.set_xlabel("Predicted"); ax.set_ylabel("LLM-consensus label")
plt.tight_layout(); plt.show()

## Input ablation
Logistic Regression with alternative inputs for each task.

In [ ]:
ablations = {
    "Expected Response Function": {"previous utterance (chosen)": ["previous_utterance"],
                                   "context + previous utterance": ["context", "previous_utterance"],
                                   "previous utterance + response": ["previous_utterance", "response"]},
    "Actual Function":            {"previous utterance + response (chosen)": ["previous_utterance", "response"],
                                   "response only": ["response"],
                                   "context + previous + response": ["context", "previous_utterance", "response"]},
    "Fulfillment":                {"context + previous + response (chosen)": ["context", "previous_utterance", "response"],
                                   "previous utterance + response": ["previous_utterance", "response"],
                                   "response only": ["response"]},
}
rows = []
for task, opts in ablations.items():
    for name, col in opts.items():
        f, _ = run_cv(task, col, ["Logistic Regression"])
        rows.append({"Task": task, "Input": name, "Macro-F1 (mean ± sd)": f"{f['Macro-F1'].mean():.3f} ± {f['Macro-F1'].std():.3f}",
                     "Accuracy": f"{f['Accuracy'].mean():.3f}"})
abl = pd.DataFrame(rows)
display(abl)

## Model comparison

In [ ]:
plot = folds_df[folds_df.Model.isin(MODEL_NAMES)].groupby(["Task", "Model"])["Macro-F1"].agg(["mean", "std"]).reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
width = 0.8 / len(MODEL_NAMES)
for i, m in enumerate(MODEL_NAMES):
    sub = plot[plot.Model == m].set_index("Task").loc[list(TASKS)]
    ax.bar(np.arange(len(TASKS)) + i * width, sub["mean"], width, yerr=sub["std"], capsize=3, label=m)
ax.set_xticks(np.arange(len(TASKS)) + width * (len(MODEL_NAMES) - 1) / 2); ax.set_xticklabels(list(TASKS))
ax.set_ylabel("Macro-F1 (mean ± sd over 5 folds)"); ax.set_ylim(0, 1); ax.legend(fontsize=8, ncol=2)
ax.set_title("Classifier comparison (grouped, stratified 5-fold cross-validation)")
plt.tight_layout(); plt.show()